## Automated RAG evaluation data generation

In [ ]:
import pandas as pd

from langchain_community.llms import Ollama
from langchain.document_loaders import DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.embeddings import OllamaEmbeddings


from ragas.testset.generator import TestsetGenerator
from ragas.testset.evolutions import simple, reasoning, multi_context

In [ ]:
pd.set_option("display.max_colwidth", None)

### Load & chunk data to create document objects

In [ ]:
loader = DirectoryLoader("./paul_graham")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=20)
documents = loader.load_and_split(text_splitter)

In [ ]:
len(documents)

### Add `filename` attribute to document 
The `filename` attribute in metadata is used to identify chunks belonging to the same document. For instance, pages belonging to the same research publication can be identified using the filename.

In [ ]:
for document in documents:
    document.metadata['filename'] = document.metadata['source']

documents[0].to_json()

### Setting up the Generator, Critic & embedding models

In [ ]:
generator_llm = Ollama(model="phi3")
critic_llm = Ollama(model="llama3")

ollama_emb = OllamaEmbeddings(
    model="nomic-embed-text",
)

print(generator_llm.invoke('Say hello'))
print(critic_llm.invoke('Say hello'))

r2 = ollama_emb.embed_query(
    "What is the second letter of Greek alphabet"
)

print(f'Embedding dimension: {len(r2)}')

### Create a test generator object

In [ ]:
generator = TestsetGenerator.from_langchain(
    generator_llm=generator_llm,
    critic_llm=critic_llm,
    embeddings=ollama_emb
)

# generate testset
testset = generator.generate_with_langchain_docs(documents, test_size=10, distributions={simple: 0.5, reasoning: 0.25, multi_context: 0.25}, raise_exceptions=False)

### Load testset as a dataframe

In [ ]:
test_df = testset.to_pandas().dropna()
test_df.head()

In [ ]:
test_df.to_csv('test_data_ollama.csv')
test_df

In [ ]:
len(test_df)